# 04 Two Option Calibration Batch (пакетная калибровка двухопционного фильтра)

Этот notebook (ноутбук) не содержит тяжелого кода. Он запускает отдельный `Python process (процесс Python)` для одного `batch (пакета)` и сразу освобождает память после завершения.

Калибруются только стратегии из `02_selected_for_calibration.csv`, то есть прошедшие `leaderboard (таблицу лидеров)`.

## Метод калибровки

Для trend strategies (трендовых стратегий) используется `period_start anchor (якорь начала периода)`: `K (страйк)` равен стартовому `PnL (прибыль и убыток)` периода плюс или минус сдвиг в долях волатильности.

Для mean-reversion candidates (кандидатов на возврат к среднему) используется центр `PnL (прибыль и убыток)`:

- `rolling_median (скользящая медиана)`;
- `MAD (медианное абсолютное отклонение)`.

Если `rolling_median + MAD (скользящая медиана + медианное абсолютное отклонение)` дает слабый результат, скрипт дополнительно проверяет `rolling_mean + rolling_std (скользящее среднее + скользящее стандартное отклонение)`.

Для mean-reversion candidates (кандидатов на возврат к среднему) сетка `T (экспираций)` включает фиксированные горизонты и множители от `half-life (периода полураспада)`. Сейчас по умолчанию используется `coarse mode (грубый режим)`. `fine mode (тонкий режим)` лучше запускать позже только по стратегиям, которые прошли первый этап.

In [1]:
from pathlib import Path
import subprocess
import sys

WORK_ROOT = Path(r"./workspace")
SCRIPT = WORK_ROOT / "two_option_calibration_batch.py"

cmd = [sys.executable, str(SCRIPT), "--mode", "summary"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Selected for calibration (отобрано для калибровки): 827
Batch size (размер пакета): 80
Batch ids (номера пакетов): 0 .. 10
Coarse trend tuples per strategy (грубые трендовые кортежи на стратегию): 216
Coarse mean reversion primary tuples per strategy (грубые основные кортежи возврата к среднему на стратегию): about 5*5*3*3 = 225 before unique half-life clipping
filter_group
put/call     370
core_only    257
call/put     118
call/call     56
put/put       26




## Запуск одного batch (пакета)

При `BATCH_SIZE (размере пакета)` 80 нужно запустить `BATCH_ID (номер пакета)` от `0` до `10`.

In [3]:
for i in range(1, 11):
    BATCH_ID = i
    BATCH_SIZE = 80
    CALIBRATION_MODE = "coarse"
    
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--mode", "batch",
        "--batch-id", str(BATCH_ID),
        "--batch-size", str(BATCH_SIZE),
        "--calibration-mode", CALIBRATION_MODE,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

{
  "batch_label": "0001",
  "start_row": 80,
  "end_row_exclusive": 160,
  "n_selected_total": 827,
  "n_strategies": 80,
  "n_grid_rows": 10003,
  "n_best_rows": 80,
  "n_best_paths_saved": 58,
  "batch_size": 80,
  "calibration_mode": "coarse"
}


{
  "batch_label": "0002",
  "start_row": 160,
  "end_row_exclusive": 240,
  "n_selected_total": 827,
  "n_strategies": 80,
  "n_grid_rows": 9538,
  "n_best_rows": 80,
  "n_best_paths_saved": 55,
  "batch_size": 80,
  "calibration_mode": "coarse"
}


{
  "batch_label": "0003",
  "start_row": 240,
  "end_row_exclusive": 320,
  "n_selected_total": 827,
  "n_strategies": 80,
  "n_grid_rows": 12846,
  "n_best_rows": 80,
  "n_best_paths_saved": 68,
  "batch_size": 80,
  "calibration_mode": "coarse"
}


{
  "batch_label": "0004",
  "start_row": 320,
  "end_row_exclusive": 400,
  "n_selected_total": 827,
  "n_strategies": 80,
  "n_grid_rows": 10316,
  "n_best_rows": 80,
  "n_best_paths_saved": 51,
  "batch_size": 80,
  "calibration_mode": "coarse

## Audit (аудит) и combine (объединение)

In [4]:
cmd = [sys.executable, str(SCRIPT), "--mode", "audit"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "selected_rows": 827,
  "processed_rows": 827,
  "processed_unique": 827,
  "missing_count": 0,
  "duplicate_count": 0,
  "missing_spec_id_ranges": [],
  "duplicate_head": []
}




In [5]:
cmd = [sys.executable, str(SCRIPT), "--mode", "combine"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "n_best_batch_files": 11,
  "n_unique_strategies": 827,
  "n_best_paths": 570,
  "output_dir": "D:\\Filter_2_option_v2\\04_two_option_calibration_output"
}




## Результаты

Все results (результаты) сохраняются на D:

`D:\Filter_2_option_v2\04_two_option_calibration_output`

Главные файлы:

- `batches/grid_batch_XXXX.csv` - полная сетка калибровки одного `batch (пакета)`;
- `batches/best_batch_XXXX.csv` - лучший вариант по каждой стратегии;
- `03_calibrated_best_by_strategy.csv` - итоговая таблица лучших параметров;
- `04_calibrated_best_paths.csv` - пути к лучшим траекториям;
- `05_manifest.json` - паспорт сборки.